# Module 5: Transformer Block & Decoder-Only Model

## Learning Objectives
- Compose attention + MLP + layernorm into a transformer decoder block
- Implement residual connections
- Run inference on toy input
- Connect to Kyro's `LlamaDecoderLayer`

## Concept: Transformer Decoder Block

Each decoder layer consists of:
1. **Multi-head self-attention** with residual connection + RMSNorm
2. **Feed-forward MLP** (SiLU-gated) with residual connection + RMSNorm

$$x' = x + \text{Attn}(\text{RmsNorm}(x))$$
$$x'' = x' + \text{MLP}(\text{RmsNorm}(x'))$$

**Kyro mapping**: `src/model/llama.rs::LlamaDecoderLayer::forward`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model=16, n_heads=2, d_ff=32, eps=1e-5):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model, eps=eps)
        self.norm2 = nn.LayerNorm(d_model, eps=eps)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.SiLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        # Self-attention with residual
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        # MLP with residual
        mlp_out = self.mlp(x)
        x = self.norm2(x + mlp_out)
        return x

# Create and run a single decoder layer
layer = TransformerDecoderLayer(d_model=16, n_heads=2, d_ff=32)
x = torch.randn(1, 4, 16)  # batch=1, seq_len=4, d_model=16
output = layer(x)
print(f'Input shape: {x.shape}, Output shape: {output.shape}')
print('Transformer decoder block forward pass successful!')

## Step 1: Chain Multiple Layers

In [ ]:
class MiniTransformer(nn.Module):
    def __init__(self, num_layers=4, d_model=16, n_heads=2, d_ff=32, vocab_size=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, n_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.lm_head(x)

# Create a mini transformer
model = MiniTransformer(num_layers=4, d_model=16, n_heads=2, d_ff=32, vocab_size=50)
input_ids = torch.randint(0, 50, (1, 8))
logits = model(input_ids)
print(f'Input: {input_ids.shape}, Logits: {logits.shape}')
print(f'Logits shape: (batch={logits.shape[0]}, seq={logits.shape[1]}, vocab={logits.shape[2]})')

## Step 2: Observe Residual Connections

In [ ]:
# Verify residual connection preserves information
x = torch.randn(1, 4, 16)
layer = TransformerDecoderLayer(d_model=16, n_heads=2, d_ff=32)

# Before and after attention
x_norm = layer.norm1(x)
attn_out, _ = layer.self_attn(x, x, x)
x_after_attn = layer.norm1(x + attn_out)

print(f'Before attention: {x.shape}')
print(f'After attention + residual: {x_after_attn.shape}')
print('Residual connection preserves gradient flow across layers.')

print('\nIn Kyro: src/model/llama.rs::LlamaDecoderLayer::forward applies input_layernorm, self_attn, residual, post_attention_layernorm, mlp, residual.')

## Exercises
1. Try different activation functions (ReLU vs SiLU vs GELU).
2. Add pre-norm vs post-norm variants.
3. Explore Kyro's `src/model/llama.rs::LlamaDecoderLayer` and `src/model/layers.rs`.